# Multi-Layer Perceptrons Workshop — Challenge 2, Part 3: Experiments and Visualisations

**Course:** CSCN8010 — Prof. David Espinosa
**Team:** Sultan Atanda, John Buni, Eche Oji, Antonio Sainz
**This notebook:** Antonio Sainz (9072844) — Part 3

Scope, as agreed in the team task split:
1. Compare ReLU vs sigmoid in the hidden layers.
2. Compare several learning rates (0.001, 0.01, 0.1 and 1.0).
3. Compare 2 hidden layers with 3 hidden layers.
4. Visualise the forward flow of activations and the backward flow of gradients through the layers.
5. Note any vanishing or exploding gradients (input for the team reflection).
6. Bonus: a diagram of how data and gradients flow through the network.

All charts use Plotly, as agreed by the team.

## 1. Shared setup (Sultan) and forward propagation (John)

The two cells below are copied unchanged from the team notebooks so that every experiment uses the same data (Two Moons, 800 training / 200 test points, standardised), the same architecture (2 → 16 → 8 → 1), the same He initialisation and the same seed.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42


class Activations:
    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def relu_derivative(z):
        return (z > 0).astype(float)

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @classmethod
    def sigmoid_derivative(cls, z):
        s = cls.sigmoid(z)
        return s * (1 - s)

    @staticmethod
    def tanh(z):
        return np.tanh(z)

    @staticmethod
    def tanh_derivative(z):
        return 1 - np.tanh(z) ** 2

    @classmethod
    def get(cls, name):
        return getattr(cls, name), getattr(cls, f"{name}_derivative")


class MoonsData:
    def __init__(self, n_samples=1000, noise=0.2, test_size=0.2, seed=SEED):
        X, y = make_moons(n_samples=n_samples, noise=noise, random_state=seed)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=seed, stratify=y
        )
        self.scaler = StandardScaler().fit(X_train)
        self.X_train = self.scaler.transform(X_train).T
        self.X_test = self.scaler.transform(X_test).T
        self.y_train = y_train.reshape(1, -1)
        self.y_test = y_test.reshape(1, -1)

    def shapes(self):
        return pd.DataFrame({
            "Array": ["X_train", "y_train", "X_test", "y_test"],
            "Shape": [self.X_train.shape, self.y_train.shape, self.X_test.shape, self.y_test.shape],
        })

    def plot(self):
        import plotly.graph_objects as go
        fig = go.Figure()
        for label, color in [(0, "#003049"), (1, "#D62828")]:
            mask = self.y_train[0] == label
            fig.add_trace(go.Scatter(
                x=self.X_train[0, mask], y=self.X_train[1, mask], mode="markers",
                name=f"Class {label}", marker=dict(color=color, size=7, opacity=0.8,
                                                   line=dict(width=0.5, color="white")),
            ))
        fig.update_layout(title="Training Data: Two Moons (standardised)", template="plotly_white",
                          xaxis_title="Feature 1", yaxis_title="Feature 2", height=480)
        fig.show()


class NetworkSetup:
    INITIALISERS = {
        "he": lambda n_in: np.sqrt(2 / n_in),
        "xavier": lambda n_in: np.sqrt(1 / n_in),
        "small": lambda n_in: 0.01,
    }

    def __init__(self, layer_dims=(2, 16, 8, 1), hidden_activation="relu",
                 output_activation="sigmoid", init="he", seed=SEED):
        self.layer_dims = list(layer_dims)
        self.L = len(self.layer_dims) - 1
        self.hidden_activation = hidden_activation
        self.output_activation = output_activation
        self.init = init
        self.seed = seed

    def initialize(self):
        rng = np.random.default_rng(self.seed)
        scale = self.INITIALISERS[self.init]
        params = {}
        for l in range(1, self.L + 1):
            n_in, n_out = self.layer_dims[l - 1], self.layer_dims[l]
            params[f"W{l}"] = rng.standard_normal((n_out, n_in)) * scale(n_in)
            params[f"b{l}"] = np.zeros((n_out, 1))
        return params

    def activation(self, l):
        name = self.output_activation if l == self.L else self.hidden_activation
        return Activations.get(name)

    def summary(self):
        rows = []
        for l in range(1, self.L + 1):
            n_in, n_out = self.layer_dims[l - 1], self.layer_dims[l]
            rows.append({
                "Layer": f"{'Output' if l == self.L else f'Hidden {l}'} (l={l})",
                "W shape": (n_out, n_in),
                "b shape": (n_out, 1),
                "Activation": self.output_activation if l == self.L else self.hidden_activation,
                "Parameters": n_out * n_in + n_out,
            })
        df = pd.DataFrame(rows)
        df.loc[len(df)] = ["Total", "", "", "", df["Parameters"].sum()]
        return df

In [2]:
def forward_propagation(X, params, setup):
    A = X
    cache = {"A0": X}

    for l in range(1, setup.L + 1):
        W = params[f"W{l}"]
        b = params[f"b{l}"]

        # Linear transformation
        Z = W @ A + b

        # Apply the layer's activation function
        activation, _ = setup.activation(l)
        A = activation(Z)

        # Save values for backpropagation
        cache[f"Z{l}"] = Z
        cache[f"A{l}"] = A

    return A, cache

In [3]:
data = MoonsData()
data.shapes()

,Array,Shape
0,X_train,"(2, 800)"
1,y_train,"(1, 800)"
2,X_test,"(2, 200)"
3,y_test,"(1, 200)"


## 2. Backpropagation and training (Eche's Part 2)

The experiments train the network with Eche's Part 2 code: `binary_cross_entropy`, `backward` and `update_parameters` are taken from his notebook with the same formulas:

- Output layer (sigmoid + cross-entropy): $dZ^{[L]} = \frac{1}{m}(A^{[L]} - Y)$
- Every layer: $dW^{[l]} = dZ^{[l]} A^{[l-1]T}$, $db^{[l]} = \sum dZ^{[l]}$
- Hidden layers: $dZ^{[l-1]} = W^{[l]T} dZ^{[l]} \odot g'(Z^{[l-1]})$
- Update: $W^{[l]} := W^{[l]} - \alpha \, dW^{[l]}$, $b^{[l]} := b^{[l]} - \alpha \, db^{[l]}$

**One adaptation:** Eche's `backward` uses the ReLU derivative directly (`Z > 0`), because his network only has ReLU hidden layers. The ReLU vs sigmoid experiment needs the derivative of whichever hidden activation is in use, so here `backward` takes the derivative from the shared setup (`setup.activation(l)`). With ReLU it gives exactly the same result as Eche's version. The gradient check below confirms the adapted version is still correct for both activations.

In [4]:
import time


def binary_cross_entropy(logits, targets):
    """Eche's loss: mean binary cross-entropy computed from the output logits Z (avoids log(0))."""
    return float(np.mean(np.logaddexp(0, logits) - targets * logits))


def backward(Y, params, cache, setup):
    """Eche's backpropagation; the hidden-layer derivative comes from the shared setup (ReLU or sigmoid)."""
    layers = len(params) // 2
    m = Y.shape[1]
    grads = {}
    dZ = (cache[f"A{layers}"] - Y) / m
    for layer in range(layers, 0, -1):
        grads[f"dZ{layer}"] = dZ
        grads[f"dW{layer}"] = dZ @ cache[f"A{layer-1}"].T
        grads[f"db{layer}"] = np.sum(dZ, axis=1, keepdims=True)
        if layer > 1:
            dA_previous = params[f"W{layer}"].T @ dZ
            _, g_prime = setup.activation(layer - 1)
            dZ = dA_previous * g_prime(cache[f"Z{layer-1}"])
    return grads


def update_parameters(params, grads, learning_rate):
    """Eche's gradient-descent update."""
    return {key: value - learning_rate * grads["d" + key] for key, value in params.items()}

**Gradient check for the adapted `backward`.** Following Eche's method, every weight and bias is nudged by ±1e-6 and the change in loss is compared with the analytic gradient, on a small random batch, for both hidden activations.

In [5]:
def check_gradients(hidden_activation, epsilon=1e-6):
    rng = np.random.default_rng(7)
    check_X = rng.normal(size=(2, 5))
    check_Y = np.array([[0., 1., 0., 1., 1.]])
    net = NetworkSetup(layer_dims=(2, 16, 8, 1), hidden_activation=hidden_activation, init="he", seed=13)
    params = net.initialize()
    for key in params:
        if key.startswith("b"):
            params[key] = rng.normal(0.2, 0.1, params[key].shape)
    _, cache = forward_propagation(check_X, params, net)
    grads = backward(check_Y, params, cache, net)
    exact, numerical = [], []
    for key, values in params.items():
        for index in np.ndindex(values.shape):
            original = values[index]
            values[index] = original + epsilon
            plus = binary_cross_entropy(forward_propagation(check_X, params, net)[1][f"Z{net.L}"], check_Y)
            values[index] = original - epsilon
            minus = binary_cross_entropy(forward_propagation(check_X, params, net)[1][f"Z{net.L}"], check_Y)
            values[index] = original
            exact.append(grads["d" + key][index])
            numerical.append((plus - minus) / (2 * epsilon))
    exact, numerical = np.array(exact), np.array(numerical)
    error = np.linalg.norm(exact - numerical) / max(np.linalg.norm(exact) + np.linalg.norm(numerical), 1e-12)
    print(f"{hidden_activation:>7}: {len(exact)} parameters checked, relative error = {error:.2e}")
    return error


for act in ["relu", "sigmoid"]:
    assert check_gradients(act) < 1e-6

   relu: 193 parameters checked, relative error = 1.81e-10
sigmoid: 193 parameters checked, relative error = 5.54e-10


The training loop below uses these functions with full-batch gradient descent on the Two Moons training set. Besides the loss, it records the size (Frobenius norm) of every layer's weight gradient at every iteration, which the gradient-flow charts in Section 6 need.

In [6]:
def train(setup, learning_rate=0.1, epochs=2000):
    """Train with Eche's backward/update. Returns the trained parameters and a history of loss and gradient norms."""
    params = setup.initialize()
    history = {"loss": [], "grad_norms": []}
    start = time.perf_counter()
    for epoch in range(epochs):
        _, cache = forward_propagation(data.X_train, params, setup)
        history["loss"].append(binary_cross_entropy(cache[f"Z{setup.L}"], data.y_train))
        grads = backward(data.y_train, params, cache, setup)
        history["grad_norms"].append([np.linalg.norm(grads[f"dW{l}"]) for l in range(1, setup.L + 1)])
        params = update_parameters(params, grads, learning_rate)
    history["seconds"] = time.perf_counter() - start
    history["grad_norms"] = np.array(history["grad_norms"])
    return params, history


def accuracy(params, setup, X, Y):
    A_out, _ = forward_propagation(X, params, setup)
    return float(((A_out >= 0.5).astype(int) == Y).mean())

## 3. Running all the experiments

Every combination of the three factors is trained for 2,000 iterations from the same starting weights (seed 42):

- Hidden activation: ReLU, sigmoid
- Depth: 2 hidden layers (2 → 16 → 8 → 1) and 3 hidden layers (2 → 16 → 8 → 8 → 1)
- Learning rate: 0.001, 0.01, 0.1, 1.0

The weight initialisation (He) and the output layer (sigmoid) stay the same, so each comparison changes only one thing at a time.

In [7]:
EPOCHS = 2000
ARCHITECTURES = {"2 hidden": (2, 16, 8, 1), "3 hidden": (2, 16, 8, 8, 1)}
ACTIVATIONS = ["relu", "sigmoid"]
LEARNING_RATES = [0.001, 0.01, 0.1, 1.0]

runs = {}
rows = []
for act in ACTIVATIONS:
    for depth_name, dims in ARCHITECTURES.items():
        for lr in LEARNING_RATES:
            net = NetworkSetup(layer_dims=dims, hidden_activation=act, output_activation="sigmoid", init="he")
            trained, hist = train(net, learning_rate=lr, epochs=EPOCHS)
            runs[(act, depth_name, lr)] = (net, trained, hist)
            rows.append({
                "Hidden activation": act,
                "Depth": depth_name,
                "Learning rate": lr,
                "Initial loss": round(hist["loss"][0], 4),
                "Final loss": round(hist["loss"][-1], 4),
                "Train accuracy": round(accuracy(trained, net, data.X_train, data.y_train), 3),
                "Test accuracy": round(accuracy(trained, net, data.X_test, data.y_test), 3),
                "Time (s)": round(hist["seconds"], 2),
            })

results = pd.DataFrame(rows)
results

,Hidden activation,Depth,Learning rate,Initial loss,Final loss,Train accuracy,Test accuracy,Time (s)
0,relu,2 hidden,0.001,0.6718,0.3867,0.864,0.905,0.55
1,relu,2 hidden,0.010,0.6718,0.2622,0.873,0.920,0.48
2,relu,2 hidden,0.100,0.6718,0.0698,0.975,0.985,0.31
3,relu,2 hidden,1.000,0.6718,0.0600,0.979,0.985,0.46
4,relu,3 hidden,0.001,0.6875,0.4589,0.834,0.890,0.58
5,relu,3 hidden,0.010,0.6875,0.2862,0.880,0.910,0.42
6,relu,3 hidden,0.100,0.6875,0.0647,0.976,0.990,0.59
7,relu,3 hidden,1.000,0.6875,0.0495,0.976,0.985,0.47
8,sigmoid,2 hidden,0.001,0.6867,0.6629,0.776,0.795,0.63
9,sigmoid,2 hidden,0.010,0.6867,0.4539,0.840,0.905,0.64


In [8]:
import plotly.graph_objects as go


def plot_loss_curves(keys, labels, title):
    fig = go.Figure()
    for key, label in zip(keys, labels):
        fig.add_trace(go.Scatter(y=runs[key][2]["loss"], mode="lines", name=label))
    fig.update_layout(title=title, xaxis_title="Iteration", yaxis_title="Training loss (binary cross-entropy)",
                      template="plotly_white", width=850, height=450)
    fig.show()

## 4. Experiments

### 4.1 ReLU vs sigmoid (2 hidden layers, learning rate 0.1)

In [9]:
plot_loss_curves(
    keys=[("relu", "2 hidden", 0.1), ("sigmoid", "2 hidden", 0.1)],
    labels=["ReLU", "Sigmoid"],
    title="Loss curve: ReLU vs sigmoid hidden layers (2 hidden layers, learning rate 0.1)",
)
results[(results["Depth"] == "2 hidden") & (results["Learning rate"] == 0.1)]

,Hidden activation,Depth,Learning rate,Initial loss,Final loss,Train accuracy,Test accuracy,Time (s)
2,relu,2 hidden,0.1,0.6718,0.0698,0.975,0.985,0.31
10,sigmoid,2 hidden,0.1,0.6867,0.2978,0.866,0.925,0.64


**Observations — activation function**
- With the same starting weights and learning rate, ReLU reached a final training loss of 0.0698 and 98.5% test accuracy, while sigmoid only reached 0.2978 and 92.5%.
- The sigmoid curve falls much more slowly. Its derivative is at most 0.25, so every sigmoid layer shrinks the error signal that reaches the earlier layers (see Section 6), and each update is small. ReLU passes the gradient through unchanged for every active neuron.
- Sigmoid is not unable to learn; it is slower. With learning rate 1.0 it reached a loss of 0.0708 and 99.0% test accuracy (table in Section 3).
- Sigmoid runs were also slower (see the time column), because the exponential is more expensive to compute than `max(0, z)`.

### 4.2 Learning rates (ReLU, 2 hidden layers)

In [10]:
plot_loss_curves(
    keys=[("relu", "2 hidden", lr) for lr in LEARNING_RATES],
    labels=[f"learning rate {lr}" for lr in LEARNING_RATES],
    title="Loss curve by learning rate (ReLU, 2 hidden layers)",
)
results[(results["Hidden activation"] == "relu") & (results["Depth"] == "2 hidden")]

,Hidden activation,Depth,Learning rate,Initial loss,Final loss,Train accuracy,Test accuracy,Time (s)
0,relu,2 hidden,0.001,0.6718,0.3867,0.864,0.905,0.55
1,relu,2 hidden,0.010,0.6718,0.2622,0.873,0.920,0.48
2,relu,2 hidden,0.100,0.6718,0.0698,0.975,0.985,0.31
3,relu,2 hidden,1.000,0.6718,0.0600,0.979,0.985,0.46


**Observations — learning rate**
- The learning rate controls the size of each gradient-descent step. At 0.001 the loss only fell from 0.6718 to 0.3867 in 2,000 iterations (90.5% test accuracy): the steps are too small to converge in that budget. At 0.01 it reached 0.2622 (92.0%).
- At 0.1 the network converged smoothly to 0.0698 (98.5%).
- At 1.0 the final loss was slightly lower (0.0600, 98.5%), but the curve is no longer smooth: the loss went *up* on 921 of the 2,000 iterations. The steps are large enough to overshoot the minimum and zig-zag around it. The gradients themselves did not grow, so this is instability from the step size, not exploding gradients.
- For this problem, 0.1 is the best trade-off between speed and stability.

### 4.3 2 vs 3 hidden layers (learning rate 0.1)

In [11]:
plot_loss_curves(
    keys=[(act, depth, 0.1) for act in ACTIVATIONS for depth in ARCHITECTURES],
    labels=[f"{act}, {depth}" for act in ACTIVATIONS for depth in ARCHITECTURES],
    title="Loss curve: 2 vs 3 hidden layers (learning rate 0.1)",
)
results[results["Learning rate"] == 0.1]

,Hidden activation,Depth,Learning rate,Initial loss,Final loss,Train accuracy,Test accuracy,Time (s)
2,relu,2 hidden,0.1,0.6718,0.0698,0.975,0.985,0.31
6,relu,3 hidden,0.1,0.6875,0.0647,0.976,0.990,0.59
10,sigmoid,2 hidden,0.1,0.6867,0.2978,0.866,0.925,0.64
14,sigmoid,3 hidden,0.1,0.7870,0.3099,0.863,0.925,0.84


**Observations — depth**
- With ReLU, adding a third hidden layer helped slightly: final loss 0.0698 → 0.0647 and test accuracy 98.5% → 99.0%.
- With sigmoid, the extra layer did not help: final loss 0.2978 → 0.3099 and test accuracy stayed at 92.5%. The deeper sigmoid network spent the first iterations on a plateau (loss still 0.678 after 100 iterations, against 0.347 for the 3-layer ReLU network).
- The clearest failure was the 3-hidden-layer sigmoid network at learning rate 0.001: the loss barely moved (0.7870 → 0.6924) and test accuracy stayed at 50%, meaning it predicted the same class for every point. Each extra sigmoid layer weakens the gradient further, so a deeper sigmoid network needs a larger learning rate to learn at all.
- Depth only pays off when the gradient can reach the early layers, which is the case with ReLU.

### 4.4 What the best model learned

The decision boundary of the ReLU network with 3 hidden layers (learning rate 0.1) shows how the trained network separates the two moons. The shaded background is the predicted probability of class 1; the points are the test set.

In [12]:
best_net, best_params, _ = runs[("relu", "3 hidden", 0.1)]
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
grid_probs, _ = forward_propagation(np.vstack([xx.ravel(), yy.ravel()]), best_params, best_net)

fig = go.Figure()
fig.add_trace(go.Contour(x=xx[0], y=yy[:, 0], z=grid_probs.reshape(xx.shape), colorscale="RdBu_r",
                         opacity=0.6, contours=dict(start=0, end=1, size=0.1), colorbar=dict(title="P(class 1)")))
for label, color in [(0, "#003049"), (1, "#D62828")]:
    mask = data.y_test[0] == label
    fig.add_trace(go.Scatter(x=data.X_test[0, mask], y=data.X_test[1, mask], mode="markers", name=f"Test class {label}",
                             marker=dict(color=color, size=7, line=dict(width=0.5, color="white"))))
fig.update_layout(title="Decision boundary: ReLU, 3 hidden layers, learning rate 0.1 (test points)",
                  xaxis_title="Feature 1", yaxis_title="Feature 2", template="plotly_white", width=750, height=600)
fig.show()

## 5. Forward flow of activations

To see how information travels forward, the untrained 3-hidden-layer network (same starting weights for both activations) is run on the training data and the distribution of every layer's activations is plotted. Each box shows the values of all neurons in that layer across the 800 training points.

In [13]:
fig = go.Figure()
for act, color in [("relu", "#1f77b4"), ("sigmoid", "#ff7f0e")]:
    net = NetworkSetup(layer_dims=ARCHITECTURES["3 hidden"], hidden_activation=act, output_activation="sigmoid", init="he")
    _, cache = forward_propagation(data.X_train, net.initialize(), net)
    for l in range(1, net.L + 1):
        name = "Output" if l == net.L else f"Hidden {l}"
        values = cache[f"A{l}"].ravel()
        q1, med, q3 = np.percentile(values, [25, 50, 75])
        fig.add_trace(go.Box(x=[name], q1=[q1], median=[med], q3=[q3], lowerfence=[values.min()],
                             upperfence=[values.max()], mean=[values.mean()], name=act, marker_color=color,
                             legendgroup=act, showlegend=(l == 1)))
fig.update_layout(title="Forward flow: activation values per layer at initialisation (3 hidden layers)",
                  xaxis_title="Layer", yaxis_title="Activation value A", boxmode="group",
                  template="plotly_white", width=850, height=480)
fig.show()

**Observations — forward flow**
- **ReLU:** about half of the activations are exactly zero in every hidden layer (49.6%, 69.2% and 51.5% in hidden layers 1–3), because ReLU switches off neurons with negative input. The neurons that stay on keep a wide range of values (standard deviation 0.67, 0.40 and 0.50), so differences between input points are carried through to the output.
- **Sigmoid:** every value is squeezed into (0, 1) around 0.5, and the spread shrinks layer by layer (standard deviation 0.216 → 0.107 → 0.131). At the output, all 800 predictions fall between 0.705 and 0.717: before training, the sigmoid network gives almost the same answer for every point, because the differences between inputs are washed out as they pass through the sigmoids.

## 6. Backward flow of gradients

The size of each layer's weight gradient, $\lVert dW^{[l]} \rVert$, shows how much of the error signal reaches that layer during backpropagation. The first chart compares the layers at the first iteration; the second follows them throughout training.

In [14]:
layer_names = ["Hidden 1", "Hidden 2", "Hidden 3", "Output"]
fig = go.Figure()
for act in ACTIVATIONS:
    first = runs[(act, "3 hidden", 0.1)][2]["grad_norms"][0]
    fig.add_trace(go.Bar(x=layer_names, y=first, name=act, text=[f"{v:.4f}" for v in first], textposition="outside"))
fig.update_layout(title="Backward flow: gradient size per layer at the first iteration (3 hidden layers, learning rate 0.1)",
                  xaxis_title="Layer (gradients flow from Output back to Hidden 1)", yaxis_title="||dW|| (log scale)",
                  yaxis_type="log", barmode="group", template="plotly_white", width=850, height=480)
fig.show()

In [15]:
fig = go.Figure()
for act, dash in [("relu", "solid"), ("sigmoid", "dot")]:
    norms = runs[(act, "3 hidden", 0.1)][2]["grad_norms"]
    for l, name in enumerate(layer_names):
        fig.add_trace(go.Scatter(y=norms[:, l], mode="lines", name=f"{act}: {name}", line=dict(dash=dash)))
fig.update_layout(title="Backward flow during training: gradient size per layer (3 hidden layers, learning rate 0.1)",
                  xaxis_title="Iteration", yaxis_title="||dW|| (log scale)", yaxis_type="log",
                  template="plotly_white", width=850, height=500)
fig.show()

**Observations — backward flow (vanishing and exploding gradients)**
- **Vanishing gradients with sigmoid:** at the first iteration the gradient shrinks at every step back through the network: output 0.3089 → hidden 3: 0.1056 → hidden 2: 0.0379 → hidden 1: 0.0105. The first hidden layer receives a gradient about 29 times smaller than the output layer, so it learns much more slowly. This is the vanishing-gradient problem: each sigmoid layer multiplies the error signal by its derivative, which is at most 0.25.
- **ReLU keeps the gradient alive:** the gradient sizes stay in the same range across all layers (hidden 1: 0.2054, hidden 2: 0.4726, hidden 3: 0.2325, output: 0.0819). The derivative of ReLU is 1 for every active neuron, so the signal is not shrunk on its way back.
- **During training** all gradients get smaller as the loss approaches its minimum, which is expected. The sigmoid layers stay ordered from largest (output) to smallest (hidden 1) for most of training.
- **Exploding gradients:** none were observed in any run; gradient sizes never grew during training. The network is shallow and He initialisation keeps the initial weights at a sensible scale. The only instability came from the learning rate of 1.0 (Section 4.2).
- **Initialisation note:** He initialisation is designed for ReLU. Using it with sigmoid is part of why the sigmoid network starts on a plateau; Xavier initialisation is the usual choice for sigmoid.

## 7. Bonus: how data and gradients flow through the network

The diagram summarises one training iteration for the 2-hidden-layer network. Blue arrows (top) are the forward pass: each layer computes $Z = W A + b$ and $A = g(Z)$ and stores both. Red arrows (bottom) are the backward pass: starting from the loss, each layer uses the stored values to compute $dZ$, $dW$ and $db$ and passes the error signal to the previous layer.

In [16]:
boxes = [("Input X", "2 features"), ("Hidden 1", "16 neurons<br>ReLU"), ("Hidden 2", "8 neurons<br>ReLU"),
         ("Output", "1 neuron<br>sigmoid"), ("Loss", "binary<br>cross-entropy")]
fig = go.Figure()
for i, (title, sub) in enumerate(boxes):
    x0 = i * 2.2
    fig.add_shape(type="rect", x0=x0, x1=x0 + 1.6, y0=0, y1=1.6, line=dict(color="#333"), fillcolor="#f2f2f2")
    fig.add_annotation(x=x0 + 0.8, y=0.8, text=f"<b>{title}</b><br>{sub}", showarrow=False, font=dict(size=13))
for i in range(len(boxes) - 1):
    x_start, x_end = i * 2.2 + 1.6, (i + 1) * 2.2
    fig.add_annotation(x=x_end, y=1.3, ax=x_start, ay=1.3, xref="x", yref="y", axref="x", ayref="y",
                       showarrow=True, arrowhead=3, arrowwidth=2, arrowcolor="#1f77b4")
    fig.add_annotation(x=x_start, y=0.3, ax=x_end, ay=0.3, xref="x", yref="y", axref="x", ayref="y",
                       showarrow=True, arrowhead=3, arrowwidth=2, arrowcolor="#d62728")
fig.add_annotation(x=4.9, y=2.1, text="<b>Forward pass:</b> Z = W·A + b, A = g(Z) (store Z and A)",
                   showarrow=False, font=dict(color="#1f77b4", size=14))
fig.add_annotation(x=4.9, y=-0.5, text="<b>Backward pass:</b> dZ → dW, db → update W, b (chain rule)",
                   showarrow=False, font=dict(color="#d62728", size=14))
fig.update_xaxes(visible=False, range=[-0.3, 10.6])
fig.update_yaxes(visible=False, range=[-0.9, 2.5])
fig.update_layout(title="Data and gradient flow in one training iteration", template="plotly_white",
                  width=950, height=380, showlegend=False)
fig.show()

## 8. Summary of findings (input for the team reflection)

| Experiment | Main result |
|---|---|
| ReLU vs sigmoid (2 hidden, learning rate 0.1) | ReLU: loss 0.0698, test accuracy 98.5%. Sigmoid: loss 0.2978, test accuracy 92.5%. Sigmoid learns more slowly. |
| Learning rate (ReLU, 2 hidden) | 0.001 too slow (loss 0.3867); 0.1 smooth and fast (0.0698); 1.0 slightly lower loss (0.0600) but the loss rose on 921 of 2,000 iterations (overshooting). |
| 2 vs 3 hidden layers (learning rate 0.1) | ReLU improved slightly (99.0% test accuracy). Sigmoid did not improve, and at learning rate 0.001 the 3-layer sigmoid network did not learn at all (50% accuracy). |
| Forward flow | Sigmoid compresses the signal layer by layer (all initial outputs between 0.705 and 0.717); ReLU keeps a wide range but switches off about half the neurons. |
| Backward flow | Vanishing gradients with sigmoid: hidden layer 1 gets a gradient about 29 times smaller than the output layer. ReLU keeps gradients in the same range. No exploding gradients observed. |

**Takeaway:** the activation function matters most for gradient stability. ReLU with He initialisation lets the error signal reach every layer, so deeper networks can still learn; sigmoid shrinks the signal at every layer, which slows learning and makes extra depth harmful unless the learning rate is raised.